In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import root_mean_squared_error

from xgboost import XGBRegressor

In [3]:
train = pd.read_csv("../data/raw/Train.csv")
test = pd.read_csv("../data/raw/Test.csv")

In [4]:
train["time"] = pd.to_datetime(train["time"])
test["time"] = pd.to_datetime(test["time"])

train["month"] = train["time"].dt.month
test["month"] = test["time"].dt.month

train["year"] = train["time"].dt.year
test["year"] = test["time"].dt.year

train["month_sin"] = np.sin(2 * np.pi * train["month"] / 12)
train["month_cos"] = np.cos(2 * np.pi * train["month"] / 12)

test["month_sin"] = np.sin(2 * np.pi * test["month"] / 12)
test["month_cos"] = np.cos(2 * np.pi * test["month"] / 12)

In [5]:
train = train.sort_values("time").reset_index(drop=True)

In [9]:
X = train.drop(columns=["sample_id", "time", "target"])
y = train["target"]

In [10]:
tscv = TimeSeriesSplit(n_splits=5)

In [11]:
scores = []

for fold, (train_idx, valid_idx) in enumerate(tscv.split(X), start=1):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = XGBRegressor(
        random_state=42,
        n_estimators=400,
        learning_rate=0.10,
        max_depth=9
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_valid)

    rmse = root_mean_squared_error(y_valid, pred)

    scores.append(rmse)

    print(f"Fold {fold}: {rmse:.4f}")

Fold 1: 0.5810
Fold 2: 0.5447
Fold 3: 0.5496
Fold 4: 0.5939
Fold 5: 0.7851


In [12]:
print(f"\nMean RMSE: {np.mean(scores):.4f}")
print(f"Std RMSE : {np.std(scores):.4f}")


Mean RMSE: 0.6109
Std RMSE : 0.0891
